In [7]:
"""
Data Analytics Project 4 – Data Visualization
DecodeLabs Industrial Training Kit | Batch 2026

Dataset: e-commerce orders (1,200 rows, 14 columns)
Goal   : Turn raw data into boardroom-ready visual insights
         following the 3 Pillars from the project guide:
           Pillar 1 – Choose with Purpose  (right chart for the question)
           Pillar 2 – Edit with Precision  (high data-ink ratio, no chartjunk)
           Pillar 3 – Tell a Story         (action titles, SCR narrative)
"""

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_PATH   = r"/Users/macbookpro/Downloads/Dataset for Data Analytics.xlsx"
OUTPUT_DIR  = Path("charts")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Global Style (High Data-Ink Ratio) ─────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor"  : "white",
    "axes.facecolor"    : "white",
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.grid"         : True,
    "grid.color"        : "#e0e0e0",
    "grid.linewidth"    : 0.6,
    "font.family"       : "sans-serif",
    "font.size"         : 11,
    "axes.titlesize"    : 13,
    "axes.titleweight"  : "bold",
    "xtick.bottom"      : False,
    "ytick.left"        : False,
})

ACCENT   = "#1565C0"   # Insight Blue (spotlight color)
NEUTRAL  = "#90A4AE"   # Muted grey for context
NEGATIVE = "#D32F2F"   # Red for bad / declining
POSITIVE = "#2E7D32"   # Green for good / growing

# ── Load & Prepare Data ────────────────────────────────────────────────────────
df = pd.read_excel(DATA_PATH, parse_dates=["Date"])
df["Year"]      = df["Date"].dt.year
df["Month"]     = df["Date"].dt.to_period("M")
df["YearMonth"] = df["Date"].dt.to_period("M").dt.to_timestamp()

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Date range : {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Products   : {sorted(df['Product'].unique())}")
print(f"Statuses   : {sorted(df['OrderStatus'].unique())}")
print()


# ══════════════════════════════════════════════════════════════════════════════
# CHART 1 – Bar Chart
# Business question: Which product generates the most revenue?
# Chart type: Horizontal bar chart (long category labels → horizontal)
# Action title states the conclusion
# ══════════════════════════════════════════════════════════════════════════════
def chart_revenue_by_product():
    rev = (
        df.groupby("Product")["TotalPrice"]
        .sum()
        .sort_values()
    )
    top_product = rev.idxmax()

    fig, ax = plt.subplots(figsize=(9, 5))

    colors = [ACCENT if p == top_product else NEUTRAL for p in rev.index]
    bars = ax.barh(rev.index, rev.values, color=colors, height=0.55)

    # Direct labels (eliminate legend cognitive load)
    for bar, val in zip(bars, rev.values):
        ax.text(
            val + rev.max() * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"${val/1e6:.2f}M",
            va="center", fontsize=10,
            color=ACCENT if rev.index[list(rev.values).index(val)] == top_product else "#555",
            fontweight="bold" if rev.index[list(rev.values).index(val)] == top_product else "normal",
        )

    ax.set_xlabel("Total Revenue (USD)", labelpad=8)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e6:.1f}M"))
    ax.set_xlim(0, rev.max() * 1.18)
    ax.set_title(
        f"{top_product}s Lead Revenue — Driving the Largest Share of Total Sales",
        pad=14,
    )
    ax.set_ylabel("")
    ax.grid(axis="y", visible=False)

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "01_revenue_by_product.png", dpi=150)
    plt.close(fig)
    print("✔ Chart 1 saved: Revenue by Product (Horizontal Bar)")


# ══════════════════════════════════════════════════════════════════════════════
# CHART 2 – Line Chart
# Business question: Is monthly revenue trending up or down?
# Chart type: Line chart (trend over time → continuous data)
# ══════════════════════════════════════════════════════════════════════════════
def chart_monthly_revenue_trend():
    monthly = (
        df.groupby("YearMonth")["TotalPrice"]
        .sum()
        .reset_index()
        .rename(columns={"TotalPrice": "Revenue"})
    )

    # Rolling 3-month average for context
    monthly["Rolling3M"] = monthly["Revenue"].rolling(3, min_periods=1).mean()

    fig, ax = plt.subplots(figsize=(12, 5))

    ax.plot(monthly["YearMonth"], monthly["Revenue"],
            color=NEUTRAL, linewidth=1.2, alpha=0.6, label="Monthly Revenue")
    ax.plot(monthly["YearMonth"], monthly["Rolling3M"],
            color=ACCENT, linewidth=2.5, label="3-Month Rolling Avg")

    # Annotate peak
    peak_idx  = monthly["Revenue"].idxmax()
    peak_row  = monthly.loc[peak_idx]
    ax.annotate(
        f"Peak\n${peak_row['Revenue']/1e3:.0f}K",
        xy=(peak_row["YearMonth"], peak_row["Revenue"]),
        xytext=(0, 22), textcoords="offset points",
        ha="center", color=ACCENT, fontsize=9, fontweight="bold",
        arrowprops=dict(arrowstyle="->", color=ACCENT, lw=1.2),
    )

    ax.set_title(
        "Monthly Revenue Shows an Upward Trend with Seasonal Spikes",
        pad=14,
    )
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
    ax.set_xlabel("")
    ax.set_ylabel("Revenue (USD)", labelpad=8)

    legend = ax.legend(frameon=False, fontsize=10)

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "02_monthly_revenue_trend.png", dpi=150)
    plt.close(fig)
    print("✔ Chart 2 saved: Monthly Revenue Trend (Line)")


# ══════════════════════════════════════════════════════════════════════════════
# CHART 3 – Stacked Bar Chart
# Business question: How do order statuses break down per product?
# Chart type: Stacked bar (composition + volume)
# ══════════════════════════════════════════════════════════════════════════════
def chart_order_status_composition():
    pivot = (
        df.groupby(["Product", "OrderStatus"])
        .size()
        .unstack(fill_value=0)
    )
    # Order statuses
    statuses    = ["Delivered", "Shipped", "Pending", "Returned", "Cancelled"]
    statuses    = [s for s in statuses if s in pivot.columns]
    status_colors = {
        "Delivered" : POSITIVE,
        "Shipped"   : ACCENT,
        "Pending"   : "#F9A825",
        "Returned"  : NEUTRAL,
        "Cancelled" : NEGATIVE,
    }
    pivot = pivot[statuses]

    fig, ax = plt.subplots(figsize=(11, 5))
    bottoms = np.zeros(len(pivot))
    for status in statuses:
        bars = ax.bar(
            pivot.index, pivot[status],
            bottom=bottoms,
            color=status_colors[status],
            label=status, width=0.55,
        )
        # Direct label if segment is large enough
        for bar, val in zip(bars, pivot[status]):
            if val > 15:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_y() + bar.get_height() / 2,
                    str(int(val)),
                    ha="center", va="center",
                    fontsize=8, color="white", fontweight="bold",
                )
        bottoms += pivot[status].values

    ax.set_title(
        "Cancellation & Return Rates Are Highest for Tablets — Action Needed",
        pad=14,
    )
    ax.set_ylabel("Order Count", labelpad=8)
    ax.set_xlabel("")
    ax.legend(loc="upper right", frameon=False, fontsize=9)
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "03_order_status_by_product.png", dpi=150)
    plt.close(fig)
    print("✔ Chart 3 saved: Order Status Composition by Product (Stacked Bar)")


# ══════════════════════════════════════════════════════════════════════════════
# CHART 4 – Scatter Plot
# Business question: Is there a relationship between items in cart & total price?
# Chart type: Scatter plot (relationship between two numeric variables)
# ══════════════════════════════════════════════════════════════════════════════
def chart_cart_vs_price_scatter():
    sample = df.sample(n=min(400, len(df)), random_state=42)
    corr   = df["ItemsInCart"].corr(df["TotalPrice"])

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(
        sample["ItemsInCart"], sample["TotalPrice"],
        alpha=0.4, color=ACCENT, s=30, linewidths=0,
    )

    # Trend line
    m, b = np.polyfit(df["ItemsInCart"], df["TotalPrice"], 1)
    xs   = np.linspace(df["ItemsInCart"].min(), df["ItemsInCart"].max(), 100)
    ax.plot(xs, m * xs + b, color=NEGATIVE, linewidth=2, label=f"Trend (r = {corr:.2f})")

    ax.set_title(
        f"More Items in Cart Weakly Predicts Higher Order Value (r = {corr:.2f})",
        pad=14,
    )
    ax.set_xlabel("Items in Cart", labelpad=8)
    ax.set_ylabel("Total Price (USD)", labelpad=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
    ax.legend(frameon=False, fontsize=10)

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "04_cart_vs_totalprice_scatter.png", dpi=150)
    plt.close(fig)
    print("✔ Chart 4 saved: Cart Size vs. Total Price (Scatter)")


# ══════════════════════════════════════════════════════════════════════════════
# CHART 5 – Grouped Bar Chart (Explanatory)
# Business question: Which referral source drives the most revenue per channel?
# Action title highlights the key insight immediately
# ══════════════════════════════════════════════════════════════════════════════
def chart_revenue_by_referral():
    ref_rev = (
        df.groupby("ReferralSource")["TotalPrice"]
        .sum()
        .sort_values(ascending=False)
    )
    top_source = ref_rev.idxmax()

    fig, ax = plt.subplots(figsize=(9, 5))
    colors = [ACCENT if s == top_source else NEUTRAL for s in ref_rev.index]
    bars   = ax.bar(ref_rev.index, ref_rev.values, color=colors, width=0.5)

    for bar, val in zip(bars, ref_rev.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            val + ref_rev.max() * 0.01,
            f"${val/1e3:.0f}K",
            ha="center", va="bottom", fontsize=10,
            color=ACCENT if ref_rev.index[list(ref_rev.values).index(val)] == top_source else "#555",
            fontweight="bold" if ref_rev.index[list(ref_rev.values).index(val)] == top_source else "normal",
        )

    ax.set_title(
        f"{top_source} Is the Top Revenue-Generating Referral Channel",
        pad=14,
    )
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
    ax.set_ylabel("Total Revenue (USD)", labelpad=8)
    ax.set_xlabel("")
    ax.set_ylim(0, ref_rev.max() * 1.15)

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "05_revenue_by_referral_source.png", dpi=150)
    plt.close(fig)
    print("✔ Chart 5 saved: Revenue by Referral Source (Bar)")


# ══════════════════════════════════════════════════════════════════════════════
# CHART 6 – Heat Map
# Business question: Which product × payment method combination has the
#                    highest cancellation rate?
# Chart type: Heat map (two categorical dimensions + numeric intensity)
# ══════════════════════════════════════════════════════════════════════════════
def chart_cancellation_heatmap():
    cancelled = df[df["OrderStatus"] == "Cancelled"]
    total     = df.groupby(["Product", "PaymentMethod"]).size()
    cancel_ct = cancelled.groupby(["Product", "PaymentMethod"]).size()
    rate      = (cancel_ct / total * 100).fillna(0).unstack(fill_value=0)

    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(rate.values, cmap="Blues", aspect="auto", vmin=0)

    # Direct cell labels
    for i in range(rate.shape[0]):
        for j in range(rate.shape[1]):
            val = rate.values[i, j]
            ax.text(
                j, i, f"{val:.0f}%",
                ha="center", va="center",
                fontsize=10,
                color="white" if val > rate.values.max() * 0.6 else "#333",
            )

    ax.set_xticks(range(len(rate.columns)))
    ax.set_xticklabels(rate.columns, rotation=30, ha="right")
    ax.set_yticks(range(len(rate.index)))
    ax.set_yticklabels(rate.index)
    ax.set_title(
        "Gift Card + Chair Orders Have the Highest Cancellation Rate",
        pad=14,
    )
    plt.colorbar(im, ax=ax, label="Cancellation Rate (%)", shrink=0.8)
    ax.grid(visible=False)

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "06_cancellation_heatmap.png", dpi=150)
    plt.close(fig)
    print("✔ Chart 6 saved: Cancellation Rate Heatmap (Product × Payment Method)")


# ══════════════════════════════════════════════════════════════════════════════
# CHART 7 – Year-over-Year Revenue Comparison (Honest Axis)
# Demonstrates the Lie Factor lesson: bar charts always start at 0
# ══════════════════════════════════════════════════════════════════════════════
def chart_yoy_revenue():
    yoy = (
        df.groupby("Year")["TotalPrice"]
        .sum()
        .reset_index()
    )
    yoy = yoy[yoy["Year"].isin([2023, 2024])]  # full calendar years only

    fig, ax = plt.subplots(figsize=(7, 5))
    colors  = [NEUTRAL, ACCENT]
    bars    = ax.bar(
        yoy["Year"].astype(str), yoy["TotalPrice"],
        color=colors, width=0.45,
    )

    # Growth annotation
    if len(yoy) == 2:
        growth = (yoy.iloc[1]["TotalPrice"] - yoy.iloc[0]["TotalPrice"]) / yoy.iloc[0]["TotalPrice"] * 100
        ax.annotate(
            f"+{growth:.1f}% YoY",
            xy=(1, yoy.iloc[1]["TotalPrice"]),
            xytext=(0, 18), textcoords="offset points",
            ha="center", fontsize=11, color=POSITIVE, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=POSITIVE),
        )

    for bar, val in zip(bars, yoy["TotalPrice"]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            val / 2,
            f"${val/1e6:.2f}M",
            ha="center", va="center",
            fontsize=12, color="white", fontweight="bold",
        )

    ax.set_ylim(0)  # Axis integrity: always start at zero
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e6:.1f}M"))
    ax.set_title("Revenue Grew Year-Over-Year (2023 → 2024)", pad=14)
    ax.set_ylabel("Total Revenue (USD)", labelpad=8)

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "07_yoy_revenue_comparison.png", dpi=150)
    plt.close(fig)
    print("✔ Chart 7 saved: YoY Revenue Comparison (Bar — Zero-Baseline Axis)")


# ══════════════════════════════════════════════════════════════════════════════
# EXECUTIVE SUMMARY DASHBOARD (SCR Narrative)
# Situation → Complication → Resolution on a single boardroom-ready figure
# ══════════════════════════════════════════════════════════════════════════════
def chart_executive_dashboard():
    fig = plt.figure(figsize=(16, 9))
    fig.patch.set_facecolor("#0D1B2A")   # Dark boardroom background

    # ── KPI boxes (top row) ────────────────────────────────────────────────
    total_rev  = df["TotalPrice"].sum()
    total_ord  = len(df)
    avg_order  = df["TotalPrice"].mean()
    cancel_rt  = (df["OrderStatus"] == "Cancelled").mean() * 100
    top_prod   = df.groupby("Product")["TotalPrice"].sum().idxmax()
    top_src    = df.groupby("ReferralSource")["TotalPrice"].sum().idxmax()

    kpis = [
        ("Total Revenue",    f"${total_rev/1e6:.2f}M",  ACCENT),
        ("Total Orders",     f"{total_ord:,}",           ACCENT),
        ("Avg Order Value",  f"${avg_order:,.0f}",       ACCENT),
        ("Cancellation Rate",f"{cancel_rt:.1f}%",        NEGATIVE),
        ("Top Product",      top_prod,                   POSITIVE),
        ("Top Channel",      top_src,                    POSITIVE),
    ]

    kpi_axes = []
    for i, (label, value, color) in enumerate(kpis):
        ax_kpi = fig.add_axes([0.01 + i * 0.165, 0.76, 0.155, 0.18])
        ax_kpi.set_facecolor("#1A2E45")
        for spine in ax_kpi.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(2)
        ax_kpi.set_xticks([])
        ax_kpi.set_yticks([])
        ax_kpi.text(0.5, 0.6, value, ha="center", va="center",
                    fontsize=16, color=color, fontweight="bold",
                    transform=ax_kpi.transAxes)
        ax_kpi.text(0.5, 0.2, label, ha="center", va="center",
                    fontsize=9, color="#B0BEC5",
                    transform=ax_kpi.transAxes)
        kpi_axes.append(ax_kpi)

    # ── Main chart: Monthly Revenue Trend ─────────────────────────────────
    monthly = (
        df.groupby("YearMonth")["TotalPrice"]
        .sum()
        .reset_index()
    )
    monthly["Roll3"] = monthly["TotalPrice"].rolling(3, min_periods=1).mean()

    ax_main = fig.add_axes([0.04, 0.08, 0.55, 0.60])
    ax_main.set_facecolor("#0D1B2A")
    ax_main.spines["bottom"].set_color("#37474F")
    ax_main.spines["left"].set_color("#37474F")
    ax_main.spines["top"].set_visible(False)
    ax_main.spines["right"].set_visible(False)
    ax_main.tick_params(colors="#B0BEC5")
    ax_main.yaxis.label.set_color("#B0BEC5")
    ax_main.xaxis.label.set_color("#B0BEC5")

    ax_main.fill_between(monthly["YearMonth"], monthly["TotalPrice"],
                         alpha=0.15, color=ACCENT)
    ax_main.plot(monthly["YearMonth"], monthly["TotalPrice"],
                 color=NEUTRAL, linewidth=1, alpha=0.5)
    ax_main.plot(monthly["YearMonth"], monthly["Roll3"],
                 color=ACCENT, linewidth=2.5)

    ax_main.set_title("Monthly Revenue Trend (3M Rolling Avg)",
                      color="white", fontsize=12, fontweight="bold", pad=10)
    ax_main.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
    ax_main.grid(color="#1E3A5F", linewidth=0.5)

    # ── Right chart: Revenue by Product ───────────────────────────────────
    rev_prod = df.groupby("Product")["TotalPrice"].sum().sort_values()
    ax_right = fig.add_axes([0.63, 0.08, 0.35, 0.60])
    ax_right.set_facecolor("#0D1B2A")
    ax_right.spines["bottom"].set_color("#37474F")
    ax_right.spines["left"].set_color("#37474F")
    ax_right.spines["top"].set_visible(False)
    ax_right.spines["right"].set_visible(False)
    ax_right.tick_params(colors="#B0BEC5")

    colors_r = [ACCENT if p == top_prod else "#37474F" for p in rev_prod.index]
    ax_right.barh(rev_prod.index, rev_prod.values, color=colors_r, height=0.55)
    ax_right.set_title("Revenue by Product",
                       color="white", fontsize=12, fontweight="bold", pad=10)
    ax_right.xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
    ax_right.grid(color="#1E3A5F", linewidth=0.5, axis="x")
    ax_right.grid(axis="y", visible=False)
    for i, (p, v) in enumerate(zip(rev_prod.index, rev_prod.values)):
        ax_right.text(v + rev_prod.max() * 0.01, i,
                      f"${v/1e3:.0f}K", va="center", fontsize=8,
                      color=ACCENT if p == top_prod else "#90A4AE")

    # ── Headline (SCR Storyteller) ─────────────────────────────────────────
    fig.text(0.5, 0.97,
             "E-Commerce Performance Dashboard  |  2023–2025",
             ha="center", va="top",
             fontsize=15, color="white", fontweight="bold")
    fig.text(0.5, 0.93,
             f"Situation: Revenue is growing YoY  ·  "
             f"Complication: {cancel_rt:.1f}% cancellation rate undermines growth  ·  "
             f"Resolution: Investigate {top_prod} & Gift Card cancellations",
             ha="center", va="top",
             fontsize=9, color="#B0BEC5")

    fig.savefig(OUTPUT_DIR / "08_executive_dashboard.png", dpi=150,
                bbox_inches="tight")
    plt.close(fig)
    print("✔ Chart 8 saved: Executive Summary Dashboard (SCR Narrative)")


# ══════════════════════════════════════════════════════════════════════════════
# RUN ALL CHARTS
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    chart_revenue_by_product()
    chart_monthly_revenue_trend()
    chart_order_status_composition()
    chart_cart_vs_price_scatter()
    chart_revenue_by_referral()
    chart_cancellation_heatmap()
    chart_yoy_revenue()
    chart_executive_dashboard()

    print()
    print(f"All charts saved to: {OUTPUT_DIR.resolve()}/")
    print("Charts generated:")
    for f in sorted(OUTPUT_DIR.glob("*.png")):
        print(f"  {f.name}")

Dataset loaded: 1,200 rows × 17 columns
Date range : 2023-01-01 → 2025-06-30
Products   : ['Chair', 'Desk', 'Laptop', 'Monitor', 'Phone', 'Printer', 'Tablet']
Statuses   : ['Cancelled', 'Delivered', 'Pending', 'Returned', 'Shipped']

✔ Chart 1 saved: Revenue by Product (Horizontal Bar)
✔ Chart 2 saved: Monthly Revenue Trend (Line)
✔ Chart 3 saved: Order Status Composition by Product (Stacked Bar)
✔ Chart 4 saved: Cart Size vs. Total Price (Scatter)
✔ Chart 5 saved: Revenue by Referral Source (Bar)
✔ Chart 6 saved: Cancellation Rate Heatmap (Product × Payment Method)
✔ Chart 7 saved: YoY Revenue Comparison (Bar — Zero-Baseline Axis)
✔ Chart 8 saved: Executive Summary Dashboard (SCR Narrative)

All charts saved to: /Users/macbookpro/Downloads/charts/
Charts generated:
  01_revenue_by_product.png
  02_monthly_revenue_trend.png
  03_order_status_by_product.png
  04_cart_vs_totalprice_scatter.png
  05_revenue_by_referral_source.png
  06_cancellation_heatmap.png
  07_yoy_revenue_comparison.p